# 05 · Control flow and loops

After this notebook you can write branches and loops the Pythonic way — truthiness, comparison chaining, `for … else`, `match` to dispatch tool calls, the walrus operator — and spot the index-based, flag-heavy loops interviewers mark down. It ends with the loop at the heart of every agent.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [04 · Dicts and sets](04_dicts_and_sets.ipynb)

## Why this matters in interviews

- Every coding round is loops and branches. Idiomatic control flow (`enumerate`, `zip`, `any`, early `return`) reads as fluency; `for i in range(len(x))` reads as a beginner.
- Truthiness bugs are real production bugs: `temperature or 0.7` silently turns a deliberate `0.0` into `0.7`.
- `match` (Python 3.10+) fits agent code naturally — dispatching on the *shape* of a tool call — and shows you know modern Python.
- The agent loop itself is a `for`/`while` with a step limit and a `break`. Same skills, most important loop in an agentic system.

## What interviewers ask

- Which values are falsy? Is `if x:` the same as `if x is not None:`?
- What does `for … else` do — when does the `else` run?
- `break` vs `continue` vs `pass`?
- What is structural pattern matching, and how is it more than a switch statement?
- What does the walrus operator `:=` do? Give a real use.
- How do you loop with an index, over two lists at once, or in reverse?
- How do you guarantee a loop — say, an agent loop — cannot run forever?

## 1 · `if` / `elif` / `else`

Branches are tested top to bottom and the **first** true one wins, so put specific conditions before general ones. A model router is a typical example.

**Predict, then run:** which model does each prompt get?

In [ ]:
def choose_model(prompt: str) -> str:
    words = len(prompt.split())
    if "code" in prompt.lower():
        return "coder-model"
    elif words > 50:
        return "large-model"
    elif words > 10:
        return "medium-model"
    else:
        return "small-model"

for p in ["Hi!", "Write code to parse this CSV file", "Summarise this: " + "word " * 60]:
    print(f"{p[:34]!r:<38} -> {choose_model(p)}")
assert choose_model("Write code " + "x " * 100) == "coder-model"       # the first true branch wins

## 2 · Truthiness

`if x:` calls `bool(x)`. These are **falsy**: `False`, `None`, zero of any numeric type (`0`, `0.0`, `0j`, `Decimal(0)`), and empty containers (`""`, `[]`, `()`, `{}`, `set()`, `range(0)`). Everything else is truthy — including `"False"`, `"0"`, `" "`, `[0]`, `[[]]` and `float("nan")`. Your own classes can define `__bool__` or `__len__`. (NumPy arrays refuse: `if np.array([1, 2]):` raises, because "true" is ambiguous for many values.)

In [ ]:
from decimal import Decimal

values = [False, None, 0, 0.0, 0j, Decimal(0), "", [], (), {}, set(), range(0),
          True, 1, -1, 0.001, "False", "0", " ", [0], [[]], (None,), float("nan")]
print(f"{'value':<16} bool(value)")
for v in values:
    print(f"{v!r:<16} {bool(v)}")
assert not any(values[:12]) and all(values[12:])

### The `or`-default bug

`and` / `or` return one of their **operands**, not a bool: `a or b` gives `a` if `a` is truthy, else `b`. That makes `value or default` a tempting one-liner — and a bug whenever a falsy value is legitimate. A caller who asks for `temperature=0.0` (deterministic output) or `max_tokens=0` silently gets the default. Test for `None` explicitly.

**Predict, then run.**

In [ ]:
def resolve_temperature_buggy(temperature=None):
    return temperature or 0.7

def resolve_temperature(temperature=None):
    return 0.7 if temperature is None else temperature

print("buggy(0.0):", resolve_temperature_buggy(0.0), "| fixed(0.0):", resolve_temperature(0.0), "| fixed():", resolve_temperature())
print("'' or 'anonymous'      ->", "" or "anonymous")
print("'Ada' and 'Ada'.upper() ->", "Ada" and "Ada".upper())
retrieved = []
print("short-circuit guard    ->", retrieved and retrieved[0])     # [] is falsy, so retrieved[0] never runs
assert resolve_temperature_buggy(0.0) == 0.7 and resolve_temperature(0.0) == 0.0

## 3 · Comparison chaining — and the `or` trap

`a < b < c` means `a < b and b < c` (with `b` evaluated once) — perfect for range checks. The classic beginner bug goes the other way: `role == "user" or "assistant"` parses as `(role == "user") or "assistant"`, which is always truthy. Use `in` with a tuple or set.

**Predict, then run:** what does the buggy check say for `role = "system"`?

In [ ]:
temperature = 1.4
print("0 <= temperature <= 2   :", 0 <= temperature <= 2)
print("1 < 2 < 3, 3 > 2 > 1 > 5:", 1 < 2 < 3, 3 > 2 > 1 > 5)

role = "system"
buggy = role == "user" or "assistant"
print("buggy check:", repr(buggy), "-> truthy:", bool(buggy))
print("correct    :", role in ("user", "assistant"))
assert bool(buggy) is True and role not in ("user", "assistant")

## 4 · The conditional expression

`value_if_true if condition else value_if_false` is an **expression**, so it fits inside an f-string, a comprehension or a function argument. Keep it to one condition; nested ternaries are hard to read — use `if`/`elif` instead.

In [ ]:
scores = [0.91, 0.42, 0.66]
labels = ["positive" if s >= 0.5 else "negative" for s in scores]
print(labels)
for n in (1, 3):
    print(f"found {n} {'result' if n == 1 else 'results'}")
assert labels == ["positive", "negative", "positive"]

## 5 · `for` loops and `range`

`for` iterates over any **iterable** — a list, a string, a dict (its keys), a file (its lines), a generator. `range(start, stop, step)` is a lazy sequence of ints: it stores three numbers, not the values, and supports `len`, indexing and O(1) `in` for ints.

In [ ]:
import sys

print("range(10, 0, -3):", list(range(10, 0, -3)))
big = range(0, 10**12, 2)
print("len:", len(big), "| big[5]:", big[5], "| 999_999 in big:", 999_999 in big, "| getsizeof:", sys.getsizeof(big), "bytes")
for ch in "RAG":
    print(ch, end=" ")
print()
for key in {"model": "x", "temperature": 0}:
    print("key:", key)
assert list(range(10, 0, -3)) == [10, 7, 4, 1] and len(big) == 5 * 10**11

## 6 · `while`, `break`, `continue`, `pass`

- `while condition:` repeats until the condition is false — for loops whose count you don't know in advance.
- `break` leaves the loop now; `continue` skips to the next iteration; `pass` does nothing (a placeholder where the syntax needs a statement).
- `while True:` with a `break` is the idiom for "loop until something happens". **Always** pair it with a limit when the exit depends on something outside your control — a network, a model.

In [ ]:
chunk_size, budget, halvings = 4096, 700, 0
while chunk_size > budget:              # unknown number of iterations
    chunk_size //= 2
    halvings += 1
print(f"chunk size {chunk_size} after {halvings} halvings")

docs = ["Refund policy ...", "", "   ", "Shipping ...", "STOP", "never reached"]
kept = []
for doc in docs:
    if doc == "STOP":
        break                           # leave the loop entirely
    if not doc.strip():
        continue                        # skip empty documents
    kept.append(doc)
print("kept:", kept)

class TodoRetriever:
    pass                                # an empty body still needs a statement

replies = iter(["", "", '{"answer": 42}'])     # a flaky model: empty twice, then a real reply
attempt, MAX_ATTEMPTS = 0, 5
while True:
    attempt += 1
    reply = next(replies)
    if reply:
        break                           # success: leave the loop
    if attempt >= MAX_ATTEMPTS:
        raise RuntimeError("model never answered")
print(f"got {reply} after {attempt} attempts")
assert chunk_size == 512 and kept == ["Refund policy ...", "Shipping ..."] and attempt == 3

## 7 · `for … else` and `while … else`

The `else` block of a loop runs when the loop finishes **without hitting `break`** — read it as "no break". It replaces the found-flag pattern in searches. Because it surprises people, a `# no break` comment is good manners.

**Predict, then run:** which of the two `else` blocks runs?

In [ ]:
docs = ["shipping is free", "refunds take 5 days", "contact support"]
ran = []

for doc in docs:
    if "refund" in doc:
        ran.append(f"found: {doc}")
        break
else:  # no break
    ran.append("else #1: no document mentions refunds")

for doc in docs:
    if "warranty" in doc:
        ran.append(f"found: {doc}")
        break
else:  # no break
    ran.append("else #2: no document mentions warranty")

found = False                           # the flag version that for-else replaces
for doc in docs:
    if "warranty" in doc:
        found = True
        break
if not found:
    ran.append("flag version: not found")
print("\n".join(ran))
assert ran == ["found: refunds take 5 days", "else #2: no document mentions warranty", "flag version: not found"]

## 8 · `match`: structural pattern matching (3.10+)

`match` compares a value against **patterns** that describe its *shape* and binds names to the parts. It is much more than a switch:

| Pattern | Matches | Example |
|---|---|---|
| literal | an equal value | `case "quit":` |
| or-pattern | any of the alternatives | `case "quit" \| "exit":` |
| capture | anything, binding a name | `case other:` |
| wildcard | anything, binding nothing | `case _:` |
| sequence | a list/tuple of that length; `*rest` takes the tail | `case ["/model", name]:` |
| mapping | a dict **containing** those keys (extra keys are ignored) | `case {"name": "search", "args": args}:` |
| class | an instance, optionally with attributes | `case str(q):`, `case Point(x=0):` |
| guard | any pattern plus an `if` condition | `case ["/temp", v] if float(v) <= 2:` |

The first matching `case` wins. A chat app's slash commands make a good first example.

In [ ]:
def handle_command(text: str) -> str:
    match text.split():
        case ["/help"]:
            return "commands: /model NAME, /temp VALUE, /reset"
        case ["/model", name]:
            return f"switched to {name}"
        case ["/temp", value] if value.replace(".", "", 1).isdigit() and float(value) <= 2:
            return f"temperature set to {float(value)}"
        case ["/temp", *_]:
            return "usage: /temp VALUE (0-2)"
        case ["/reset" | "/clear"]:
            return "history cleared"
        case [cmd, *_] if cmd.startswith("/"):
            return f"unknown command {cmd}"
        case _:
            return "(a normal chat message)"

for t in ["/help", "/model qwen2.5:3b", "/temp 0.2", "/temp 9", "/clear", "/oops now", "hello there"]:
    print(f"{t!r:<22} -> {handle_command(t)}")
assert handle_command("/temp 9") == "usage: /temp VALUE (0-2)" and handle_command("/clear") == "history cleared"

### Dispatching tool calls with mapping patterns

When a model decides to call a tool, your code receives a dict like `{"name": "search", "args": {...}}`. Mapping patterns check the keys you care about, pull out the values and can check their types (`str(q)`, `int(k)`), all in one readable place. (Careful: `int(k)` also matches `True`, because `bool` is a subclass of `int` — [notebook 01](01_variables_types_numbers.ipynb).)

**Predict, then run:** which case handles the weather call that carries an extra `"id"` key? And the search call whose `top_k` is the string `"3"`?

In [ ]:
def dispatch(call) -> str:
    match call:
        case {"name": "search", "args": {"query": str(query), "top_k": int(k)}}:
            return f"search {query!r}, top {k}"
        case {"name": "search", "args": {"query": str(query)}}:
            return f"search {query!r}, top 5 (default)"
        case {"name": "get_weather", "args": {"city": str(city), **rest}}:
            return f"weather in {city}, unit={rest.get('unit', 'celsius')}"
        case {"name": "final_answer", "args": {"text": text}}:
            return f"FINAL: {text}"
        case {"name": str(name)}:
            return f"error: unknown tool {name!r}"
        case _:
            return "error: malformed tool call"

calls = [
    {"name": "search", "args": {"query": "refund policy", "top_k": 3}},
    {"name": "search", "args": {"query": "refund policy", "top_k": "3"}},       # wrong type -> falls through
    {"id": "call_1", "name": "get_weather", "args": {"city": "Paris", "unit": "F"}},
    {"name": "send_email", "args": {}},
    ["not", "a", "dict"],
]
for c in calls:
    print(dispatch(c))
assert dispatch(calls[1]) == "search 'refund policy', top 5 (default)" and dispatch(calls[2]) == "weather in Paris, unit=F"

### The capture-pattern trap

A bare name in a `case` is a **capture**, not a comparison: it matches anything and *rebinds* the name. To compare against a constant, use a literal or a **dotted name** (an enum member, `module.CONSTANT`). Python rejects a bare capture that is not the last case — which catches some of these mistakes, but not this one.

In [ ]:
from enum import Enum

EXPECTED_TOOL = "search"
match "calculator":
    case EXPECTED_TOOL:                    # captures! EXPECTED_TOOL is now "calculator"
        matched = True
print("matched:", matched, "| EXPECTED_TOOL is now:", repr(EXPECTED_TOOL))

class ToolName(Enum):
    SEARCH = "search"
    CALCULATOR = "calculator"

match "calculator":
    case ToolName.SEARCH.value:            # a dotted name is a value pattern: compared with ==
        routed = "search"
    case ToolName.CALCULATOR.value:
        routed = "calculator"
print("with dotted names:", routed)
assert EXPECTED_TOOL == "calculator" and routed == "calculator"

## 9 · The walrus operator `:=`

`name := expression` assigns **and** returns the value, so you can test something and keep it in one step. Three honest uses: reading a stream in blocks until it is empty, using a regex match you just tested, and computing a value once inside a comprehension's filter.

In [ ]:
import io
import re

stream = io.StringIO("chunk-1|chunk-2|chunk-3|")
received = []
while (block := stream.read(8)):              # stops when read() returns '' (falsy)
    received.append(block)
print("blocks:", received)

review = "Score: 8/10 - clear and correct."
if (m := re.search(r"Score: (\d+)/10", review)):
    print("parsed score:", int(m.group(1)))

texts = ["short", "a much longer document text", "", "medium text here"]
long_counts = [n for t in texts if (n := len(t.split())) >= 2]
print("word counts >= 2:", long_counts)
assert received == ["chunk-1|", "chunk-2|", "chunk-3|"] and long_counts == [5, 3]

## 10 · Loop idioms that replace index loops

| Instead of | Write | Why |
|---|---|---|
| `for i in range(len(xs)): x = xs[i]` | `for x in xs:` | say what you mean |
| index **and** item | `for i, x in enumerate(xs):` | no off-by-one |
| two lists by index | `for q, a in zip(qs, answers):` | stops at the shortest (or `strict=True`) |
| `for i in range(len(xs) - 1, -1, -1)` | `for x in reversed(xs):` | no index arithmetic |
| a manual total / best | `sum(...)`, `max(..., key=...)` | built-ins, running in C |
| flag + `break` search | `any(...)`, `next((x for x in xs if ...), None)`, or `for … else` | one expression |
| removing items while iterating | build a new list | mutating the list you loop over skips items |

In [ ]:
chunks = ["intro", "refund rules", "shipping", "refund exceptions"]
scores = [0.2, 0.9, 0.1, 0.7]

for rank, (chunk, score) in enumerate(sorted(zip(chunks, scores), key=lambda p: -p[1]), start=1):
    print(f"{rank}. {chunk:<18} {score:.1f}")

total = sum(scores)                                    # accumulate
best = max(zip(chunks, scores), key=lambda p: p[1])    # search for the best
has_refund = any("refund" in c for c in chunks)        # "is there one?" as one expression
first_refund = next((c for c in chunks if "refund" in c), None)
print(f"total={total:.1f}  best={best}  any refund? {has_refund}  first={first_refund!r}")
print("newest first:", list(reversed(chunks)))
assert best == ("refund rules", 0.9) and first_refund == "refund rules"

**Predict, then run:** this loop removes every `2` from the list it is iterating over. What is left?

In [ ]:
nums = [1, 2, 2, 3, 2]
for n in nums:
    if n == 2:
        nums.remove(n)               # shifts the remaining items under the loop's feet
print("removed inside the loop:", nums)
buggy_result = list(nums)

nums = [1, 2, 2, 3, 2]
nums = [n for n in nums if n != 2]   # build a new list instead
print("filtered into a new list:", nums)
assert buggy_result == [1, 3, 2] and nums == [1, 3]

In [ ]:
import timeit

import matplotlib.pyplot as plt

data = list(range(100_000))

def index_loop():
    total = 0
    for i in range(len(data)):
        total += data[i]
    return total

def direct_loop():
    total = 0
    for x in data:
        total += x
    return total

def builtin_sum():
    return sum(data)

times = {f.__name__: min(timeit.repeat(f, number=5, repeat=3)) / 5 * 1000 for f in (index_loop, direct_loop, builtin_sum)}
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh(list(times), list(times.values()), color=["#adb5bd", "#74c0fc", "#4c6ef5"])
ax.invert_yaxis()
ax.set_title("Summing 100,000 ints three ways"); ax.set_xlabel("milliseconds (lower is better)")
plt.show()
print({k: f"{v:.2f} ms" for k, v in times.items()})
assert index_loop() == direct_loop() == builtin_sum() and times["builtin_sum"] < times["index_loop"]

Direct iteration beats indexing a little; the built-in beats both by a lot, because its loop runs in C. Readability is the main reason to prefer the idioms — speed is a bonus.

## 11 · Putting it together: the agent loop

An agent is control flow around a model: ask the model what to do, `match` on the shape of its reply, run the tool, record the result, and loop — with a hard step limit so a confused model cannot loop forever. `for step in range(MAX_STEPS)` gives the limit for free, `break` exits on the final answer, and the loop's `else` handles "ran out of steps". Here the "model" is a scripted list of replies, so control flow is the only thing on show; the real thing is in [agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb).

In [ ]:
from matplotlib.patches import FancyBboxPatch

def draw_pipeline(steps, title=None):
    """Boxes joined by arrows, left to right."""
    fig, ax = plt.subplots(figsize=(min(2.3 * len(steps), 11), 1.7))
    for i, s in enumerate(steps):
        x = i * 2.3
        ax.add_patch(FancyBboxPatch((x, 0), 1.9, 1, boxstyle="round,pad=0.06", fc="#e7f0ff", ec="#3b5bdb"))
        ax.text(x + 0.95, 0.5, s, ha="center", va="center", fontsize=9)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + 2.28, 0.5), xytext=(x + 1.98, 0.5), arrowprops=dict(arrowstyle="->", lw=1.6))
    ax.set_xlim(-0.2, len(steps) * 2.3); ax.set_ylim(-0.25, 1.25); ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()

draw_pipeline(["ask the model\n(next reply)", "match on\nthe reply's shape", "run the tool", "record\nthe result",
               "loop again\n(max MAX_STEPS)\nor break"], title="The agent loop: a for-loop with a match, a break and an else")

In [ ]:
import operator

OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}

def calculator(args):
    a, op, b = args["expression"].split()          # e.g. "30 - 12"
    return str(OPS[op](float(a), float(b)))

TOOLS = {"search": lambda args: "Refunds are accepted within 30 days of delivery.", "calculator": calculator}

def run_agent(replies, max_steps=5):
    transcript = []
    for step in range(1, max_steps + 1):
        reply = next(replies)                        # stands in for a model call
        match reply:
            case {"type": "final", "text": str(text)}:
                answer = text
                break
            case {"type": "tool_call", "name": str(name), "args": dict(args)} if name in TOOLS:
                result = TOOLS[name](args)
                transcript.append(f"step {step}: {name}({args}) -> {result!r}")
            case _:
                transcript.append(f"step {step}: could not use reply {reply!r}")
    else:  # no break: the model never produced a final answer
        answer = "Stopped: step limit reached without a final answer."
    return answer, transcript

scripted = iter([
    {"type": "tool_call", "name": "search", "args": {"query": "refund window"}},
    {"type": "tool_call", "name": "calculator", "args": {"expression": "30 - 12"}},
    {"type": "final", "text": "You have 18 days left to request a refund."},
])
answer, transcript = run_agent(scripted)
print("\n".join(transcript))
print("answer:", answer)
assert answer.startswith("You have 18 days") and len(transcript) == 2

## Try it yourself

**1.** Write `first_long_doc(docs, min_words)` that returns the first document with at least `min_words` words, or `None` — using `for … else`, no flag variable.

In [ ]:
# Solution — try it yourself first, then run this
def first_long_doc(docs, min_words):
    for doc in docs:
        if len(doc.split()) >= min_words:
            break
    else:  # no break
        return None
    return doc

docs = ["hi", "refunds take five business days", "shipping is free"]
print(first_long_doc(docs, 3), "|", first_long_doc(docs, 10))
assert first_long_doc(docs, 3) == "refunds take five business days" and first_long_doc(docs, 10) is None

**2.** Write `parse_top_k(text)` with `match`: `"/top_k N"` returns `N` as an int when it is a whole number from 1 to 20; anything else returns `None`.

In [ ]:
# Solution — try it yourself first, then run this
def parse_top_k(text: str):
    match text.split():
        case ["/top_k", n] if n.isdigit() and 1 <= int(n) <= 20:
            return int(n)
        case _:
            return None

print([parse_top_k(t) for t in ["/top_k 5", "/top_k 50", "/top_k five", "/top_k", "/top_k 3 4"]])
assert parse_top_k("/top_k 5") == 5 and parse_top_k("/top_k 50") is None and parse_top_k("/top_k 3 4") is None

**3.** Rewrite this index loop with `zip` and `enumerate`, computing the accuracy and the indices of the wrong predictions:
```python
correct = 0; wrong = []
for i in range(len(gold)):
    if gold[i] == pred[i]: correct += 1
    else: wrong.append(i)
```

In [ ]:
# Solution — try it yourself first, then run this
gold = ["positive", "negative", "neutral", "positive"]
pred = ["positive", "neutral", "neutral", "negative"]
accuracy = sum(g == p for g, p in zip(gold, pred, strict=True)) / len(gold)   # True counts as 1
wrong = [i for i, (g, p) in enumerate(zip(gold, pred)) if g != p]
print(f"accuracy={accuracy:.0%} wrong={wrong}")
assert accuracy == 0.5 and wrong == [1, 3]

**4.** Show that the step limit works: run `run_agent` with a "model" that *always* asks for another search (`itertools.repeat`). The loop must stop after 5 steps with the "Stopped" answer.

In [ ]:
# Solution — try it yourself first, then run this
from itertools import repeat

stuck_model = repeat({"type": "tool_call", "name": "search", "args": {"query": "refund"}})   # infinite iterator
answer, transcript = run_agent(stuck_model, max_steps=5)
print(answer, "| steps taken:", len(transcript))
assert answer.startswith("Stopped") and len(transcript) == 5

## Say it in an interview

- **Truthiness (30 s):** "`if x` calls `bool(x)`: `None`, `False`, zeros and empty containers are falsy, everything else is truthy. `or` returns an operand, so `x or default` replaces legitimate falsy values like `0` or `0.0` — for optional parameters I write `default if x is None else x`."
- **`for … else`:** the `else` runs only if the loop was not broken out of — a search without a flag. `while … else` works the same way.
- **`match`:** structural pattern matching on shape — sequences, mappings (extra keys ignored), classes, guards, captures. Ideal for dispatching tool calls or commands. A bare name in a `case` *captures*; compare constants with dotted names.
- **Walrus:** `while (chunk := f.read(n)):`, `if (m := re.search(...)):` — assign and test in one step. Don't use it where a plain assignment reads better.
- **Idioms:** iterate directly; `enumerate`, `zip(strict=True)`, `reversed`, `sorted(key=)`, `any`/`all`, `next(gen, default)`; never mutate a list while looping over it.
- **Agent loops:** "Every loop whose exit depends on a model gets a hard step limit — `for step in range(MAX_STEPS)` with a `break` on the final answer and an `else` that reports running out of steps."

## Next

- [06 · Functions: arguments, scope, closures and lambdas](06_functions_args_scope_lambda.ipynb).
- [Agent loop from scratch](../15_agentic_ai/01_agent_loop_from_scratch.ipynb) — the loop above with a real model and real tools.
- [Function calling](../08_llm_apps/04_function_calling.ipynb) — where tool-call dicts come from.
- [Warm-up problems](../02_python_problems/01_warmup_problems.ipynb) — practise the idioms.
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 5: control flow) · the [agentic_ai_flow course](../../agentic_ai_flow/index.html).